# Plot maps showing catchment characteristics and features and monitoring locations

In [ ]:
# Imports
import os
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import geopandas as gpd
import contextily as ctx
import pandas as pd

In [ ]:
# General settings

# Whether to save plots
save_flag = True

# Catchement and asset characterisation files location
path_catchment_and_asset_characterisation = '../1_catchment_and_asset_characterisation'

# Gauge index
path_index_telemetry_data = '../index_telemetry_data.csv'
df_index = pd.read_csv(path_index_telemetry_data)

# Folder to save plots in
path_output_folder = 'outputs/1_feature_maps'
if not os.path.isdir(path_output_folder):
    os.makedirs(path_output_folder)

# Plot settings
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10
})

## Catchment features

In [ ]:
# Shape file paths
shp_path_catchment = f'{path_catchment_and_asset_characterisation}/catchment_outline/Study_area.shp'
shp_path_links = f'{path_catchment_and_asset_characterisation}/links/Links.shp'
shp_path_nodes = f'{path_catchment_and_asset_characterisation}/nodes/Nodes.shp'
shp_path_2d_zones_elevation = f'{path_catchment_and_asset_characterisation}/elevation/2D Zones elevation.shp'
shp_path_2d_zones_infiltration = f'{path_catchment_and_asset_characterisation}/infiltration/2D Zones infiltration.shp'

### Plot catchment outline

In [ ]:
# Function to create plot with catchment outline

def plot_catchment_outline(
        ax,
        shp_path_catchment: str,
        map_source = ctx.providers.OpenStreetMap.Mapnik,     # Other options include ctx.providers.OpenTopoMap, ctx.providers.CartoDB.Positron, ctx.providers.CartoDB.Voyager,
        line_colour: str = 'dimgrey',
        line_width: float = 1.5,
        line_style: str = '-.',
        zoom = 'auto',          # Determines map resolution
        scale: float = 1,       # Determines map scale
        reset_extent=True
):
    # Load catchment shapefile
    gdf_catchment = gpd.read_file(
        shp_path_catchment, columns=['NAME', 'geometry'])
    
    # Reproject to Web Mercator (required for contextily basemaps)
    gdf_catchment = gdf_catchment.to_crs(epsg=3857)
    
    # Get catchment centre and calculate extents for specified scale
    xmin, ymin, xmax, ymax = gdf_catchment.total_bounds
    centre_x = (xmin + xmax) / 2
    centre_y = (ymin + ymax) / 2
    width = xmax - xmin
    height = ymax - ymin
    half_catchment_size = max(width, height) * scale / 2
    ax.set_xlim(centre_x - half_catchment_size, centre_x + half_catchment_size)
    ax.set_ylim(centre_y - half_catchment_size, centre_y + half_catchment_size)

    # Keep map square
    ax.set_aspect('equal')

    # Add to plot
    gdf_catchment.plot(
        ax=ax,
        facecolor='none',
        edgecolor=line_colour,
        linewidth=line_width,
        linestyle=line_style
    )
    # Add basemap
    ctx.add_basemap(
        ax,
        source=map_source,
        crs=gdf_catchment.crs,
        zoom=zoom,
        reset_extent=reset_extent
    )
    # Remove tick marks
    ax.set_xticks([])
    ax.set_yticks([])
    

In [ ]:
# Plot catchment outline only
fig, ax_catchment = plt.subplots(1, 1, figsize=(6, 6))
plot_catchment_outline(
    ax_catchment,
    shp_path_catchment,
    map_source=ctx.providers.OpenTopoMap,
    line_colour='black',
    scale=1.1,
    zoom=12,
    line_width=3
    )
ax_catchment.set_title('Case study area')

plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Catchment outline.png', dpi=600, bbox_inches='tight')

plt.close(fig)

### Plot links

In [ ]:
# Load 'links' geodataframe from shape file
gdf_links = gpd.read_file(shp_path_links)

# Reproject geodataframe to Web Mercator
gdf_links = gdf_links.to_crs(epsg=3857)

In [ ]:
# Extract sewer system links for plotting
# (links with system_type of 'combined', 'foul' or 'storm' and link_type not of 'River reach')
gdf_sewer = gdf_links[
    (gdf_links['system_typ'].isin(['combined', 'foul', 'storm'])) &
    (gdf_links['link_type'] != 'River reach')
]

# Categorise system types for legend
system_category_map = {
    'storm': 'Storm',
    'combined': 'Combined / foul',
    'foul': 'Combined / foul'
}
gdf_sewer['system_category'] = gdf_sewer['system_typ'].map(system_category_map)

# Define colour map for plotting
colour_map_sewer_system_type = {
    'Combined / foul': 'black',
    'Storm': 'red'
}

# Define line width map for plotting
line_width_map_sewer_system_type = {
    'Combined / foul': 1.3,
    'Storm': 2.3
}

In [ ]:
# Extract fluvial / overland system links for plotting
# (links with system_type of 'other' or 'overland' AND/OR link_type of 'River reach')

gdf_fluvial = gdf_links[
    (gdf_links['system_typ'].isin(['other', 'overland'])) |
    (gdf_links['link_type'] == 'River reach')
]

# Categorise link types to simplify legend
link_category_map = {
    'Conduit': 'Conduit / orifice',
    'Culvert inlet': 'River / culvert / control structure',
    'Culvert outlet': 'River / culvert / control structure',
    'Flap valve': 'River / culvert / control structure',
    'Orifice': 'Conduit / orifice',
    'Pump': 'River / culvert / control structure',
    'River reach': 'River / culvert / control structure',
    'Screen': 'River / culvert / control structure',
    'Sluice': 'River / culvert / control structure',
    'Weir': 'River / culvert / control structure',
}
gdf_fluvial['link_category'] = gdf_fluvial['link_type'].map(link_category_map)

# Define colour map for plotting
colour_map_link_category = {
    'Conduit / orifice': 'black',
    'River / culvert / control structure': 'blue',
}

# Define line width map for plotting
line_width_map_link_category = {
    'Conduit / orifice': 1.5,
    'River / culvert / control structure': 1.5,
}

In [ ]:
# Plot sewer system and fluvial/overland system on adjacent plots

# Initialise figure
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Add base map and catchment outline
for ax in axes:
    plot_catchment_outline(
        ax,
        shp_path_catchment,
        map_source=ctx.providers.CartoDB.Voyager,
        line_width=0,
        scale=1
    )

# Add fluvial/overland system to first plot and sewer system to second plot
for ax, gdf, colour_map, colour_map_column, title, line_width_map, alpha in zip(
    [axes[0], axes[1]],
    [gdf_fluvial, gdf_sewer],
    [colour_map_link_category, colour_map_sewer_system_type],
    ['link_category', 'system_category'],
    ['a)', 'b)'], # or ['Fluvial / overland system', 'Sewer system']
    [line_width_map_link_category, line_width_map_sewer_system_type],
    [1, 0.4]
):   
    gdf.plot(
        ax=ax,
        edgecolor=gdf[colour_map_column].map(colour_map),
        linewidth=gdf[colour_map_column].map(line_width_map),
        cmap=colors.ListedColormap(list(colour_map.values())),
        column=colour_map_column,
        legend=True,
        alpha=alpha
    )
    ax.set_title(title)
    legend = ax.get_legend()            # Get the geopandas legend (not a standard Matplotlib legend)
    legend.set_bbox_to_anchor((1, 0))   # Anchor to bottom right corner
    legend.set_loc('lower right')       # Set anchor

plt.tight_layout()

plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Catchment link features.png', dpi=600, bbox_inches='tight')

plt.close(fig)

### Plot elevation from 2D zones mesh, alongside catchment location

In [ ]:
# Load geodataframe from shape file
gdf_2d_zones = gpd.read_file(shp_path_2d_zones_elevation)

# Reproject geodataframe to Web Mercator
gdf_2d_zones = gdf_2d_zones.to_crs(epsg=3857)

In [ ]:
# Plot catchment location and elevations on adjacent plots

# Initialise figure
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Add base map and catchment outline
plot_catchment_outline(
    axes[0],
    shp_path_catchment,
    line_width=5,
    line_style='-',
    line_colour='red',
    scale=50,
    reset_extent=False
    )
plot_catchment_outline(
    axes[1],
    shp_path_catchment,
    map_source=ctx.providers.CartoDB.Voyager,
    line_width=0,
    scale=1
    )

# Add titles
axes[0].set_title('a)')
axes[1].set_title('b)')


# Plot elevations

# Create an axis for the colorbar (bottom-right overlay)
cax = fig.add_axes([0.76, 0.15, 0.185, 0.03])  # [left, bottom, width, height]

# Add 2D zones
# gdf_2d_zones.loc[0:1000].plot(
gdf_2d_zones.plot(
    ax=axes[1],
    linewidth=0,
    cmap='viridis',
    column='elevation2',
    legend=True,
    cax=cax,          # Explicitly control colorbar placement
    legend_kwds={
        'label': 'Elevation (m)',
        'orientation': 'horizontal'
    },
)

plt.tight_layout()
plt.show()

plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Catchment location and elevations.png', dpi=600, bbox_inches='tight')

plt.close(fig)

### Plot runoff coefficient and horton model infiltration parameters from 2D zones mesh

In [ ]:
# Load geodataframe from shape file
gdf_2d_zones = gpd.read_file(shp_path_2d_zones_infiltration)

# Reproject geodataframe to Web Mercator
gdf_2d_zones = gdf_2d_zones.to_crs(epsg=3857)

In [ ]:
# Plot runoff parameters
for column, label, save_name, title in zip(
    ['runoff_vol', 'initial_in', 'limiting_i', 'decay_fact', 'recovery_f', 'runoff_coe'],
    ['Infiltration model', 'Horton initial infiltration (mm/hr)', 'Horton limiting infiltration (mm/hr)', 'Horton decay factor (/hr)', 'Horton recovery factor (/hr)', 'Runoff coefficient (-)'],
    ['infiltration model', 'initial infiltration', 'limiting infiltration', 'decay factor', 'recovery factor', 'runoff coeefficient'],
    ['a)', 'c)', 'd)', 'e)', 'f)', 'b)']
):
    # Initialise figure
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))

    # Add base map and catchment outline
    plot_catchment_outline(ax, shp_path_catchment, map_source=ctx.providers.CartoDB.Voyager, line_width=0)

    is_numeric = pd.api.types.is_numeric_dtype(gdf_2d_zones[column])
    if is_numeric:
        legend_kwds = {
            'label': label,
            'orientation': 'horizontal'
        }
        # Create an axis for the colorbar (bottom-right overlay)
        cax = fig.add_axes([0.52, 0.15, 0.37, 0.03])  # [left, bottom, width, height]
    else:
        legend_kwds = {}
        cax=None

    # Add 2D zones
    gdf_2d_zones.plot(
        ax=ax,
        linewidth=0,
        cmap=plt.colormaps['viridis'].reversed(),
        column=column,
        legend=True,
        cax=cax,          # Explicitly control colorbar placement
        legend_kwds=legend_kwds,
        missing_kwds={
            'color': 'white',   # fill colour for missing values
        }
    )

    if not is_numeric:
        legend = ax.get_legend()            # Get the geopandas legend (not a standard Matplotlib legend)      
        legend.set_bbox_to_anchor((1, 0))   # Anchor to bottom right corner
        legend.set_loc('lower right')       # Set anchor

    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    # Save

    if save_flag:
        fig.savefig(f'{path_output_folder}/Catchment runoff {save_name}.png', dpi=600, bbox_inches='tight')

    plt.close(fig)

## Sensor locations

In [ ]:
# Convert gauge index geodataframe
gdf_gauges = gpd.GeoDataFrame(
    df_index,
    geometry=gpd.points_from_xy(df_index['locationLongitude'], df_index['locationLatitude']),
    crs="EPSG:4326"
)

# Reproject to match basemap CRS
gdf_gauges = gdf_gauges.to_crs(epsg=3857)

In [ ]:
# Plot sensor locations 

# Initialise figure
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

# Define marker style and colour for each gauge type
marker_styles = {
    'Sewer': {'marker': '+', 'color': 'teal'},
    'Borehole': {'marker': 'o', 'color': 'purple'},
    'Rain': {'marker': 's', 'color': 'blue'},
    'Fluvial': {'marker': 'x', 'color': 'red'},
    'Highway gully': {'marker': '^', 'color': 'black'}
}

# Add base map
plot_catchment_outline(
    ax,
    shp_path_catchment,
    map_source=ctx.providers.CartoDB.Voyager,
    line_width=1,
    scale=1
    )

# Add titles
# ax.set_title('Sensor locations')

# Plot locations for each gauge type
for gauge_type, group in gdf_gauges.groupby('gaugeType'):
    style = marker_styles.get(
        gauge_type,
        {'marker': 'o', 'color': 'black'}  # fallback
    )
    group.plot(
        ax=ax,
        marker=style['marker'],
        color=style['color'],
        markersize=40,
        label=gauge_type
    )

# Add legend
ax.legend(
    title='Gauge type',
    loc='lower right',
    bbox_to_anchor=(1, 0)
    )

plt.tight_layout()
plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Gauge locations.png', dpi=600, bbox_inches='tight')

plt.close(fig)